# Import Libraries

In [ ]:
# =======================
#     Core Libraries
# =======================
import os
import pickle
import numpy as np
import pandas as pd

# =======================
#   Data Visualization
# =======================
import matplotlib.pyplot as plt
import seaborn as sns

# =======================
#      Preprocessing
# =======================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE

# =======================
#       ML Models
# =======================
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression

# =======================
#   Evaluation Metrics
# =======================
from sklearn.metrics import accuracy_score, f1_score

# Basic Tasks

In [ ]:
# Load Dataset
df = pd.read_csv("./datasets/thyroid_disease.csv")

In [ ]:
# Display First 5 Rows
print("Head of the dataset:\n")
df.head()

In [ ]:
# Shape of the Dataset
print("Shape of the dataset:\n")
df.shape

In [ ]:
# Info about Dataset
print("Info about the dataset:\n")
df.info()

In [ ]:
# Check for Duplicates and Remove if Any
duplicates = df.duplicated().sum()
print(f"Total duplicate rows: {duplicates}\n")

if duplicates > 0:
    df = df.drop_duplicates()
    print("? Duplicates removed.")

In [ ]:
# Check and Handle Missing Values
print(f"Shape of dataset before removing missing values:{df.shape} \n")

missing = df.isnull().mean() * 100
print("Missing Value Percentage:\n", missing)

for col in df.columns[df.isnull().any()]:
    if missing[col] < 5:
        df.dropna(subset=[col], inplace=True)
        print(f"? Dropped rows with missing values in '{col}' (<5%).")
    else:
        fill_value = df[col].mode()[0] if df[col].dtype == 'object' else df[col].median()
        df[col].fillna(fill_value, inplace=True)
        print(f"?? Filled missing values in '{col}' with {'mode' if df[col].dtype == 'object' else 'median'}.")

print(f"Shape of dataset after removing missing values:{df.shape} \n")

In [ ]:
# Class Distribution (Assuming last column is target)
target_col = df.columns[-1]

# Plotting the Class Distribution
plt.figure(figsize=(6,4))
sns.countplot(x=target_col, data=df)
plt.title('Class Distribution')
plt.xlabel('Class')
plt.ylabel('Count')
plt.show()

# Checking for Imbalance
class_counts = df[target_col].value_counts()
print(f"Class Distribution:\n{class_counts}")

# If class distribution is imbalanced, suggest techniques
if class_counts.min() / class_counts.max() < 0.1:  # If the ratio between smallest and largest class is <10%
    print("?? Class distribution is imbalanced. Consider applying imbalance handling techniques (like SMOTE, undersampling, or oversampling).")
else:
    print("? Class distribution is balanced. No need to apply imbalance techniques.")


# Data Preprocessing

## 1. Handling Data Types

In [ ]:
# Iterate through columns and check for 'object' type
for col in df.columns:
    if df[col].dtype == 'object':  # Check if the column is of object type
        print(f"Unique values in '{col}':")
        print(df[col].unique())
        print()  # For better readability

In [ ]:
# 1. Label Encoding for binary columns (including 'Recurred')
binary_columns = ['Gender', 'Smoking', 'Hx Smoking', 'Hx Radiothreapy', 'Focality', 'Recurred']

label_encoder = LabelEncoder()
for col in binary_columns:
    df[col] = label_encoder.fit_transform(df[col])
    print(f"? Label Encoded '{col}'.")

# 2. One-Hot Encoding for multi-class columns
multi_class_columns = ['Thyroid Function', 'Physical Examination', 'Adenopathy', 
                       'Pathology', 'Risk', 'T', 'N', 'M', 'Stage', 'Response']

df = pd.get_dummies(df, columns=multi_class_columns, drop_first=True)
print("\n? One-Hot Encoding applied to multi-class columns.")

# 3. Convert bool columns to int (0/1)
bool_cols = df.select_dtypes(include='bool').columns

for col in bool_cols:
    df[col] = df[col].astype(int)

# 4. Move 'Recurred' column to the last
recurred = df.pop('Recurred')
df['Recurred'] = recurred
print("\n? 'Recurred' column moved to the last.")

# 5. Dataset is ready!


In [ ]:
df.info()

## 3. Standardization data

In [ ]:
# Separate features and target
X = df.drop('Recurred', axis=1)  # Features
y = df['Recurred']               # Target

# Initialize StandardScaler
scaler = StandardScaler()

# Fit and transform X
X_scaled = scaler.fit_transform(X)

# Convert X_scaled back to DataFrame with original column names
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# Merge scaled X and y back
df_scaled = pd.concat([X_scaled, y.reset_index(drop=True)], axis=1)

print("? StandardScaler applied to feature columns only (Recurred untouched).")


# Modeling

In [ ]:
# separate features and class
X = df_scaled.iloc[:,:-1]
y = df_scaled.iloc[:, -1]

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Get the feature names from X for future use in Feature selection techniques
feature_names = X.columns.tolist()

# Define models
models = {
    'KNN': KNeighborsClassifier(),
    'Logistic Regression': LogisticRegression(),
    'Decision Tree': DecisionTreeClassifier(),
    'SVM': SVC(),
    'Naive Bayes': GaussianNB()
}

In [ ]:
print("Evaluating models with all features: \n")

# Separate dictionaries for each metric
all_features_model_accuracies = {}
all_features_model_f1_scores = {}

# Loop through each model
for model_name, model_instance in models.items():
    model_instance.fit(X_train, y_train)
    y_pred = model_instance.predict(X_test)

    # Save metrics
    accuracy = round(accuracy_score(y_test, y_pred) * 100, 2)
    f1 = round(f1_score(y_test, y_pred, average='weighted') * 100, 2)

    # Add metrics to the respective dictionaries
    all_features_model_accuracies[model_name] = accuracy
    all_features_model_f1_scores[model_name] = f1

# Print results
print("Baseline model accuracies (with all features):")
print(all_features_model_accuracies)

print("\nBaseline model F1 scores (with all features):")
print(all_features_model_f1_scores)

# Pipeline + pickle

In [ ]:
# Create the directory if it doesn't exist
os.makedirs('./models', exist_ok=True)

# Assuming the 'models' dictionary already exists and the model is trained
thyroid_disease_model = models['Logistic Regression']

# Save the model to the specified path
with open('./models/thyroid_disease_model.pkl', 'wb') as file:
    pickle.dump(thyroid_disease_model, file)

print("? Logistic Regression model saved to './models/thyroid_disease_model.pkl'")

# Result

In [ ]:
# Dynamically extract models and values
model = list(all_features_model_accuracies.keys())
accuracies = list(all_features_model_accuracies.values())
f1_scores = list(all_features_model_f1_scores.values())

# Validate that the dictionaries have the same keys
if set(all_features_model_accuracies.keys()) != set(all_features_model_f1_scores.keys()):
    raise ValueError("Dictionaries must have the same models as keys")

# --- Plot 1: Bar Chart for Accuracies ---
plt.figure(figsize=(8, 5))
x = np.arange(len(models))
plt.bar(x, accuracies, color='skyblue')
plt.xlabel('Models')
plt.ylabel('Accuracy (%)')
plt.title('Model Performance: Accuracy')
plt.xticks(x, models, rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# --- Plot 2: Bar Chart for F1 Scores ---
plt.figure(figsize=(8, 5))
x = np.arange(len(models))
plt.bar(x, f1_scores, color='lightcoral')
plt.xlabel('Models')
plt.ylabel('F1 Score (%)')
plt.title('Model Performance: F1 Scores')
plt.xticks(x, models, rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()